In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy.stats as stats
from functools import reduce
from pathlib import Path
import random
import missingno as msno
from scipy.stats import pearsonr
import warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor 
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import PowerTransformer
import ipywidgets as widgets
from IPython.display import display
import matplotlib.patches as patches
import calendar
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
import requests
import json
import datetime
import hashlib
import os


Vamos a automatizar la carga, limpieza y unificacion de los datos de la planta de cerveza que estan repartidos en varios archivos Excel.

1. Busca los archivos de los totalizadores de distintos años y los abre hoja por hoja.

2. Valida que cada dia tenga la estructura corecta (25 registros por jornadas) y descarta automaticamente los dias corruptos o con datos apilados para evitar que arruinen el analisis posterior.

3. Normaliza las columnas de fecha (DIA) y hora (HORA), elimina solapamientos eligiendo los datos mas recientes, renombra las columnas agregando el nombre de su hoja de origen para no perder contexto, y cruza todo en una unica tabla maestra llamada df_final.

In [ ]:
carpeta_actual = Path.cwd().resolve()
raiz_proyecto = next(
    (ruta for ruta in (carpeta_actual, *carpeta_actual.parents)
     if (ruta / 'notebooks' / 'eda.ipynb').is_file()),
    None,
)
if raiz_proyecto is None:
    raise FileNotFoundError('Ejecutá el notebook desde la raíz del proyecto o notebooks.')

ruta_unificado = raiz_proyecto / 'data' / 'dataset_unificado_inicial.csv'
if not ruta_unificado.is_file():
    raise FileNotFoundError(f'Falta {ruta_unificado}. Ejecutá el EDA hasta el paso 5.1.')

# Leemos el dataset completo generado en el EDA.
df_final = pd.read_csv(ruta_unificado, low_memory=False, parse_dates=['DIA'])

print(f'Cargado: {df_final.shape[0]} filas y {df_final.shape[1]} columnas desde {ruta_unificado}.')

In [ ]:
# Mostrar las primeras 5 filas del DataFrame final
display(df_final.head())

# Mostrar información general (columnas, tipos de datos y valores nulos)
df_final.info()

# <font color= "red"> Limpieza de datos

## <font color= "orange"> Tratamiento de los registros inválidos detectados: días incompletos, totales imposibles y días sin reinicio del totalizador.

Si bien anteriormente se hizo una limpieza archivo por archivo,  ahora vamos a hacer una limpieza mas fina sobre el resultado final para asegurarnos de que el dataset quede sin baches y defasajes que el otro filtro no logro ver.

¿Por que? Anteriormente se lee hoja por hoja. Ahi se borraron aquellos dias que tenian mas de 25 registros en esa hoja especifica. Cuando termina de leer todas las hojas por separado se cruzan multiples planillas distintas. Al hacer esto, se generan nuevas combinaciones de filas o se suman registros que individualmente tenian 25 en su hoja, pero que al cruzarse con otra hoja del mismo dia terminan desfasandose y sumando 26 o mas.

Es por este motivo que vamos a eliminar aquellos dias que no llegan al numero de 25 registros

In [ ]:
#Agrupamos por día y contamos cuantas horas (registros) tiene cada uno
conteo_por_dia=df_final.groupby('DIA').size()

#filtramos solo los que tengan menos de 25 registros
dias_incompletos=conteo_por_dia[conteo_por_dia<25]

print(f"Total de días incompletos encontrados: {len(dias_incompletos)}")
print("\nDetalle de los días y sus registros:")
print(dias_incompletos)

In [ ]:
#filtramos solo los que tengan menos de 25 registros
dias_con_mas=conteo_por_dia[conteo_por_dia>25]

print(f"Total de días con mas de 25 registros encontrados: {len(dias_con_mas)}")
print("\nDetalle de los días y sus registros:")
print(dias_con_mas)

In [ ]:
plt.figure(figsize=(14, 5))

#Contamos los registros por día
conteo_por_dia=df_final.groupby('DIA').size()

plt.plot(conteo_por_dia.index, conteo_por_dia.values, marker='.', linestyle='none', color='b', alpha=0.6, label='Registros diarios')
plt.axhline(y=24, color='r', linestyle='--', label='Esperado (24 registros)')

plt.title('Control de Registros por Día (Detección de Días Incompletos)')
plt.xlabel('Fecha')
plt.ylabel('Cantidad de Horas / Registros')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

Podemos ver que la inmensa mayoria de los puntos se concentran de manera uniforme en los 25 registros diarios. Esto nos confirma visualmente cual es el comportamiento regular y esperado de la adquisicion de datos de la planta.

Se observan algunos puntos aislados por debajo de la linea base. Esto indica que en esas fechas especificas faltaron horas de registro debido a fallos o interrupciones en la planta.

Tambien se pueden observar picos muy por encima del limite normal (como el que ronda los 50 registros). Esto evidencia días con datos duplicados, solapamiento de archivos o informacion acumulada por error.

Como pudimos ver hay muy pocos dias que tienen menos de 25 registros, nosotros decidimos descartarlos por completo. Esto se debe a que son muy pocos los datos eliminados y al tener una muestra muy grande no afecta para el entrenamiento. 
No podriamos usar algun metodo de imputacion ya que se estaria inventando un comportamiento operativo falso, pudiendo alterar metricas criticas de eficiencia energetica.
Mantener dias con baches introduce ruido y sesgos que degradan la capacidad de generalizacion del modelo. Es preferible perder una fraccion emnor de muestras que contaminar el dataset con informacion precisa

In [ ]:
print(f"Filas totales antes del filtro: {df_final.shape[0]}")

#contamos la cantidad de registros por cada día
conteo_horas=df_final.groupby('DIA').size()

#detectamos los días con menos de 25 registros
dias_incompletos=conteo_horas[conteo_horas < 25].index

if not dias_incompletos.empty:
    print(f"Se detectaron {len(dias_incompletos)} días con menos de 24 registros:")
    for d in dias_incompletos:
        print(f"   - {d.strftime('%Y-%m-%d')}: {conteo_horas[d]} registros")
    
    #filtramos el DataFrame conservando únicamente los días completos
    df_final=df_final[~df_final['DIA'].isin(dias_incompletos)].copy()
    print("\n Días incompletos eliminados correctamente.")
else:
    print(" Todos los días tienen los 25 registros completos.")

print(f" Filas resultantes tras el filtro: {df_final.shape[0]}")

In [ ]:
print(f"Filas totales antes del filtro: {df_final.shape[0]}")

#contamos la cantidad de registros por cada día
conteo_horas=df_final.groupby('DIA').size()

#detectamos los días con mas de 25 registros
dias_imposibles=conteo_horas[conteo_horas>25].index

if not dias_imposibles.empty:
    print(f"Se detectaron {len(dias_imposibles)} días con mas de 25 registros:")
    for d in dias_imposibles:
        print(f"   - {d.strftime('%Y-%m-%d')}: {conteo_horas[d]} registros")
    
    #filtramos el DataFrame conservando únicamente los días completos
    df_final=df_final[~df_final['DIA'].isin(dias_imposibles)].copy()
    print("\n Días con mas de 25 registros eliminados correctamente.")
else:
    print(" Todos los días tienen los 25 registros completos.")

print(f" Filas resultantes tras el filtro: {df_final.shape[0]}")

Los datos provienen de medidores de energia que funcionan como contadores acumulativos. El consumo de una hora se calcula restando el valor actual menos el anterior. Si un dia tiene registros de mas, significa que hubo solapamientos o saltos de datos corruptos, lo que rompe por completo la secuencia lógica y matematica de la acumulacion.

Intentar adivinar que fila duplicada borrar o como arreglar un dia sobrecargado a mano o mediante promedios podria generar problemas. 
* Podriamos falsear el consumo real de energia de la planta cervecera. 
* Los modelos de ML aprenderian patrones ficticios o ruido en lugar del comportamiento operativo real. 
Los algoritmos de prediccion energetica necesitan una estructura matricial totalmente regular y homogenea. Permitir dias con 25 registros y otros con 26 o 50 rompe la periocidad temporal y confunde al modelo durante el entrenamiento. 

Por este motivo tambien decidimos eliminarlos. Perder esas pocas jornadas no afecta en absoluto el volumen de datos para entrenar el modelo, pero garantiza que el 100% de los datos que conservemos sean confiables.


In [ ]:
columnas_numericas=df_final.select_dtypes(include=['float64', 'int64']).columns

df_medianoche=df_final[df_final['HORA'].astype(str).str.startswith('00:00:00')].copy()

reporte_fallas = {}

#Iteramos sobre las filas de la medianoche para buscar valores mayores a 0
for index, row in df_medianoche.iterrows():
    fecha_str = row['DIA'].strftime('%Y-%m-%d')
    fallas_del_dia = []
    
    for col in columnas_numericas:
        valor = row[col]
        #Verificamos que no sea NaN (nulo) y que sea mayor estricto que 0
        if pd.notna(valor) and valor > 0:
            fallas_del_dia.append(f"{col}: {valor}")
            
    # Si encontramos al menos un totalizador que no se reinició ese día, lo guardamos
    if fallas_del_dia:
        reporte_fallas[fecha_str] = fallas_del_dia

#Imprimimos el reporte de forma legible
print("REPORTE DE TOTALIZADORES QUE NO SE REINICIARON A LAS 00:00\n")

if not reporte_fallas:
    print("Todos los totalizadores se reiniciaron a 0 correctamente en la medianoche.")
else:
    for fecha, lista_fallas in reporte_fallas.items():
        print(f"Fecha: {fecha}")
        for falla in lista_fallas:
            print(f"  - {falla}")
        print("-" * 40)

Como hay algunos dias que el totalizador no empezo en cero, vamos a descartarlos por completo para evitar corromper la variable objetivo y la acumulabilidad del sistema. Sin embargo establecimos un umbral solo para eliminar aquellas que se excedan de este mismo. 

Inicialmente no usamos ningun umbral, sin embargo pasabamos de 28 mil muestras a tan solo 400. Como nos quitaba mas del 90% de los datos decidimos poner un umbral. Inicialmente el umbral era de 500, pero lo fuimos bajando y llegamos a ver que muchos de ellos arrancan a las 00:00 con valores extremadamente chicos (1,2,3,4,5 o 6 kW) y por este motivo es que eliminaba todos.

Podemos ver en base a este analisis que no representan una falla de reinicio. Que un dia arranquemos a las 00:00 con 6kW es ruido tipico de los sensores o una pequeña carga residual, no representa que el totalizador se rompio y arrastro el acumulador del dia anterior. Una falla real de reinicio es cuando el contador arranca en 15 mil o 20 kW.

In [ ]:
df_final['HORA'] = df_final['HORA'].astype(str).str.strip()

# Identificamos de forma automática la columna objetivo del totalizador de frío
col_frio = [c for c in df_final.columns if 'Frio' in c and 'Kw' in c][0]
print(f"Columna objetivo detectada: {col_frio}")

# Separamos registros de medianoche (00:00) y de cierre (23:59)
df_medianoche = df_final[df_final['HORA'].str.startswith('00:00:00')].copy()
df_cierre = df_final[df_final['HORA'].str.startswith('23:59:00')].copy()

# Aseguramos tipo numérico para el cálculo
s_medianoche = pd.to_numeric(df_medianoche[col_frio], errors='coerce')
s_cierre = pd.to_numeric(df_cierre[col_frio], errors='coerce')

# Calculamos la escala en el primer 70% cronológico (Train) para evitar Data Leakage
fechas_ordenadas = np.sort(df_final['DIA'].unique())
fecha_corte_train = fechas_ordenadas[int(len(fechas_ordenadas) * 0.70)]

mediana_cierre_train = s_cierre[df_cierre['DIA'] <= fecha_corte_train].median()
q95_medianoche_train = s_medianoche[df_medianoche['DIA'] <= fecha_corte_train].quantile(0.95)

# Definimos el umbral relativo a la escala del sensor:
# Por ejemplo, el 20% de un cierre diario normal (suficiente para tolerar ruido de arranque a las 00:00,
# pero implacable si el totalizador arrastró el acumulado de 24 hs del día anterior).
umbral_reinicio = 0.20 * mediana_cierre_train

print(f"Mediana de cierre diario (23:59) en Train: {mediana_cierre_train:.2f}")
print(f"Percentil 95 en medianoche (00:00) en Train: {q95_medianoche_train:.2f}")
print(f"Umbral adaptativo de reinicio (20% del cierre típico): {umbral_reinicio:.2f}")

# Detectamos los días donde a las 00:00 se supera el umbral relativo
dias_sin_reinicio = df_medianoche.loc[s_medianoche > umbral_reinicio, 'DIA'].unique()

print(f"Días detectados sin reinicio en el totalizador: {len(dias_sin_reinicio)}")

# Filtramos el DataFrame principal conservando únicamente los días sanos
df_final_limpio = df_final[~df_final['DIA'].isin(dias_sin_reinicio)].copy()

print(f"Filas originales: {df_final.shape[0]}")
print(f"Filas tras eliminar días sin reinicio: {df_final_limpio.shape[0]}")

Nosotros decidimos eliminar aquellas que realmetne no se reiniciaron, pero quedarnos con aquellas que si bien no son 0 exacto al reiniciarse hay un cierto valor que representa ruido instrumental propia de la lectura de los totalizadores.

Elegimos quedarnos con aquellas que arranquen con 6kW o menos, que si observamos en una planta que consume 35000kW a diarios, arrancar con 6 o 7 unidades representaria menos del 0.05% del consumo diario.

Filtramos el dataset, solo nos interesa quedarnos con la hora 23:59 ya que estos datos los utilizaremos para predecir

In [ ]:
df_final_limpio['DIA'] = pd.to_datetime(df_final_limpio['DIA']).dt.normalize()
df_final_limpio = df_final_limpio.sort_values(['DIA', 'HORA']).reset_index(drop=True)

# Filtramos para quedarnos estrictamente con el último registro de la jornada (23:59)
hora_corte = '23:59:00' 
df_diario = df_final_limpio[df_final_limpio['HORA'] == hora_corte].copy()

# NUEVO: Ponemos DIA como índice y forzamos frecuencia diaria continua ('D').
# Los días que fueron eliminados antes aparecerán como filas de NaNs, 
# garantizando que cada fila sea exactamente 1 día calendario.
df_diario = df_diario.drop(columns=['HORA'], errors='ignore').set_index('DIA').sort_index()
df_diario = df_diario.asfreq('D')

print(f"Grilla diaria continua creada: {df_diario.shape[0]} días calendario totales.")
print(f"Días válidos presentes: {df_diario['Frio (Kw) (Consolidado EE)'].notna().sum()}")

In [ ]:
# Definimos la columna objetivo real y exacta
col_objetivo = 'Frio (Kw) (Consolidado EE)'

# Buscamos de forma explícita el día calendario siguiente.
# Si mañana no tiene cierre, el Target queda vacío y nunca toma el dato de pasado mañana.
frio_calendario = df_diario[col_objetivo].copy()
dia_siguiente = df_diario.index + pd.Timedelta(days=1)
df_diario['Target_Frio_Dia_Siguiente'] = frio_calendario.reindex(dia_siguiente).to_numpy()

print(f"Variable objetivo creada respetando saltos de calendario.")
print(f"Días con Target válido listos para cuando terminemos los Lags: {df_diario['Target_Frio_Dia_Siguiente'].notna().sum()}")

Target_Frio_Dia_Siguiente: Se excluye totalemtne de las features. Esta es la etiqueta que el modelo tiene que predecir. Si la metemos como feature dentro de las variables rpedictoras, cometemos data leakage porque le estariamos dando al modelo la respuesta del futuro antes de que intente adivinarla.

Frio (Kw) (Consolidado EE): Este ahora si se usa como feature. Es el totalizador de frio de hoy. Como estamos parado hoy y queremos predecir el consumo de mañana, el dato de hoy es una variable de entrada fundamental porque le aporta la inercia y memoria que la planta trae arrastrando.

## <font color= "orange"> Correcciones pt1

In [ ]:
# Reemplazamos cualquier cadena de texto que represente un error de Excel/CSV por NaN
df_diario = df_diario.replace(
    ['#VALUE!', '#DIV/0!', '#REF!', '#N/A', '#NAME?', '#NULL!', '#NUM!', ''], 
    np.nan
)

# Convertimos todas las columnas a numérico (DIA ya está a salvo porque es el índice)
for col in df_diario.columns:
    df_diario[col] = pd.to_numeric(df_diario[col], errors='coerce')

print("¡Errores de fórmula convertidos a NaN con éxito!")

In [ ]:
# Verificar filas duplicadas (mismos valores en todo el registro diario)
filas_duplicadas = df_diario.duplicated().sum()
print(f"ANÁLISIS DE DUPLICADOS")
print(f"Cantidad de filas duplicadas exactas: {filas_duplicadas}")


**Objetivo:** Limpiar el dataset diario de sensores (`df_diario`) eliminando únicamente aquellos sensores que han dejado de reportar, **sin borrar por error** los sensores que fueron incorporados tarde (nuevos).

### Pasos que ejecuta este bloque

1. **Conversión temporal**
   - Copia el DataFrame original en `df_temp` (no modifica `df_diario` hasta el final).
   - Convierte la columna `DIA` de microsegundos (`unit='us'`) a fechas `datetime` reales con `errors='coerce'`.
   - Establece `DIA` como índice y verifica el rango temporal resultante.

2. **Cálculo de nulos por tramos**
   Para cada sensor se calcula el porcentaje de valores nulos en:
   - El **primer 30%** de la serie (`nulos_inicio`) → detecta sensores que no existían al principio.
   - El **último 30%** de la serie (`nulos_fin`) → detecta sensores que han dejado de reportar.

3. **Clasificación de sensores (2 reglas)**

   | Regla | Condición | Acción |
   |---|---|---|
   | 🪦 **Sensor muerto** | `nulos_fin > 0.95` | Se elimina del dataset |
   | 🆕 **Sensor nuevo** | `nulos_inicio > 0.50` **y** `nulos_fin < 0.20` | Se **protege** (no se borra) |

4. **Eliminación selectiva**
   - Solo se eliminan las columnas del vector `sensores_muertos`.
   - Los sensores nuevos se conservan intactos.

5. **Verificación de seguridad**
   - Se lanza un `assert` para confirmar que ningún sensor marcado como "nuevo" fue eliminado por error.

6. **Reasignación final**
   - El DataFrame depurado se guarda de nuevo en `df_diario`.

### ⚠️ Consideraciones

- Los umbrales (30%, 95%, 50%, 20%) son heurísticos y ajustables según el dominio.

### 📤 Output esperado
- Lista de sensores muertos eliminados.
- Lista de sensores nuevos protegidos.
- Número final de variables (`df_procesado.shape[1]`).

In [ ]:
df_temp = df_diario.copy()

# Filtramos las filas que son días válidos (no huecos del calendario)
dias_validos_mask = df_temp['Frio (Kw) (Consolidado EE)'].notna()
df_temp_validos = df_temp[dias_validos_mask]

# --- CORRECCIÓN ANTI-LEAKAGE: Miramos SOLO el conjunto de Train (primer 70%) ---
n_train_validos = int(len(df_temp_validos) * 0.70)
df_train_audit = df_temp_validos.iloc[:n_train_validos]

print(f"Rango total del dataset: {df_temp_validos.index.min().date()} hasta {df_temp_validos.index.max().date()}")
print(f"Rango evaluado para auditar sensores (Train 70%): {df_train_audit.index.min().date()} hasta {df_train_audit.index.max().date()}")

# Cálculo vectorizado de nulos por tramo (SOLO dentro de Train)
n = len(df_train_audit)
corte_30 = int(n * 0.30)
corte_70 = int(n * 0.70)

nulos_inicio = df_train_audit.iloc[:corte_30].isnull().mean()
nulos_fin    = df_train_audit.iloc[corte_70:].isnull().mean()
nulos_total  = df_train_audit.isnull().mean()

# Máscaras booleanas con las reglas de negocio evaluadas en Train
mask_muertos = nulos_fin > 0.95
mask_nuevos  = (~mask_muertos) & (nulos_inicio > 0.50) & (nulos_fin < 0.20)
mask_zombies = (~mask_muertos) & (~mask_nuevos) & (nulos_total > 0.70)

vector_sensores_muertos = df_train_audit.columns[mask_muertos].tolist()
vector_sensores_nuevos_salvados = df_train_audit.columns[mask_nuevos].tolist()
vector_sensores_zombies = df_train_audit.columns[mask_zombies].tolist()

print(f"\n--- SENSORES MUERTOS A ELIMINAR ({len(vector_sensores_muertos)}) ---")
print(vector_sensores_muertos)

print(f"\n--- SENSORES NUEVOS PROTEGIDOS ({len(vector_sensores_nuevos_salvados)}) ---")
print(vector_sensores_nuevos_salvados)

print(f"\n--- SENSORES DUDOSOS ({len(vector_sensores_zombies)}) ---")
print(vector_sensores_zombies)

# Aplicamos la eliminación de columnas a todo el dataset
df_diario = df_temp.drop(columns=vector_sensores_muertos)
print(f"\nDataset depurado con éxito (sin fuga de datos). Variables finales: {df_diario.shape[1]}")

Si una variable deja de existir por completo en años recientes, no la podemos usar para entrenar modelos que van a predecir el futuro (porque e el futuro ese sensor ya no está). Esas columnas, dependiendo del criterio, suelen descartarse para el modelado.

Ahora si se incorporaron variables tarde (no estaban en 2020 ni 2021 pero si en 2022, 2023, etc) no se deben borrar bajo ningun punto de vista. Son valiosas porque reflejan la planta moderna y van a estar presentes en el futuro. 

Vamos a filtrar el dataframe por fecha para quedarnos unicamente con el periodo donde la planta opero con la configuracion de sensores actual. Si hacemos el recorte temporal antes de evaluar los nulos de las columnas, logramos que los sensores que se instalaron tarde (los que no estaban en 2020 pero si en 2023 por ejemplo) dejen de arrastrar años enteros de nulos artificiales. 

In [ ]:
#Filtramos el DataFrame para graficar SOLAMENTE las columnas de los sensores muertos
df_sensores_muertos_vis = df_temp[vector_sensores_muertos]

# 3. Generamos el mapa de calor exclusivo para estos 61 sensores
plt.figure(figsize=(16, 12))

sns.heatmap(
    df_sensores_muertos_vis.isnull().T, 
    cbar=True, 
    cmap='magma_r', # Un mapa de calor contrastado para ver bien el corte
    yticklabels=True, 
    cbar_kws={'label': '¿Es nulo? (1 = Sí, 0 = No)'}
)

plt.title(f"Auditoría Visual: Comportamiento de los {len(vector_sensores_muertos)} Sensores Detectados como 'Muertos'\n(Verificando que realmente dejen de registrar al final del período)", fontsize=14, fontweight='bold')
plt.xlabel("Línea de Tiempo (2020 - 2023)", fontsize=11)
plt.ylabel("Sensores / Variables a Eliminar", fontsize=11)
plt.tight_layout()
plt.show()

Podemos ver como no solo hay variables que nunca tuvieron datos en toda la linea de tiempo, sino tambien podemos ver como hay sensores que murieron a mitad de camino. Estos decidimos borrarlos ya que no nos van a servir para predecir la variable objetivo.

## <font color= "Red"> Feature Engineering

Empecemos con Dia de la Smena, Mes del Año e Indicador de fin de semana (binario). Estas son las mas sencillas porque no dependen de valores numericos de sensores (no sufren por los nulos ni por la escala) y se pueden extraer directamente del indice de fecha y hora del dataframe original.

In [ ]:
#Verificamos que el índice sea datetime
if not isinstance(df_diario.index, pd.DatetimeIndex):
    df_diario.index = pd.to_datetime(df_diario.index)

# Extracción de variables de calendario
df_diario['Anio'] = df_diario.index.year
df_diario['Mes'] = df_diario.index.month
df_diario['Dia_Semana'] = df_diario.index.dayofweek  # Lunes = 0, Domingo = 6
df_diario['Es_Fin_De_Semana'] = df_diario['Dia_Semana'].isin([5, 6]).astype(int) # 1 si es Sáb/Dom, 0 sino

print("¡Variables temporales creadas con éxito!")
print(df_diario[['Mes', 'Dia_Semana', 'Es_Fin_De_Semana']].head(12))

Ahora creamos Variables de Lag (rezagos) y agregaciones temporales (promedios moviles).
* Variables de Lag: Le permiten al modelo saber que paso exactamente ayer, hace dos dias o hace 7. En una cerveceria, el consumo o comportamiento de frio de hoy esta fuertemente atado a lo que pasó en dias anteriores.

* Promedios Moviles(rolling): Suavizan el ruido diario y capturan la tendencia de corto plazo (por ejemplo, el promedio de consumo de los ultimos 7 dias)

Estas variables deben calcularse sobre el DataFrame completo antes de separarlo. Como solo miran hacia atras en el tiempo (usando datos pasados para predcir el futuro), no generan fuga de datos.

In [ ]:
target = 'Target_Frio_Dia_Siguiente'
col_objetivo = 'Frio (Kw) (Consolidado EE)'

# Seleccionamos las columnas numéricas que no sean el target ni las variables de calendario
cols_para_features = df_diario.select_dtypes(include=[np.number]).columns.drop(
    [target, 'Anio', 'Mes', 'Dia_Semana', 'Es_Fin_De_Semana'], 
    errors='ignore'
)

print(f"Creando variables temporales para {len(cols_para_features)} columnas numéricas...")

# Guardamos en un diccionario para unir de golpe y evitar el PerformanceWarning de fragmentación en Pandas
nuevas_cols_lag = {}

for col in cols_para_features:
    # Como el índice es diario continuo ('D'), shift(1) es exactamente ayer y shift(7) hace 7 días calendario
    nuevas_cols_lag[f'{col}_Lag_1'] = df_diario[col].shift(1)
    nuevas_cols_lag[f'{col}_Lag_7'] = df_diario[col].shift(7)
    
    # min_periods=1 permite calcular el promedio aunque falte algún día aislado en la ventana de 7 días
    nuevas_cols_lag[f'{col}_MediaMov_7d'] = (
        df_diario[col].rolling(window=7, min_periods=1, closed='left').mean()
    )

# Unimos todas las columnas creadas al DataFrame principal
df_diario = pd.concat([df_diario, pd.DataFrame(nuevas_cols_lag, index=df_diario.index)], axis=1)

# AHORA SÍ: Eliminamos las filas que eran huecos del calendario (donde hoy no hubo datos)
# y las filas donde mañana fue un hueco (Target_Frio_Dia_Siguiente es NaN)
filas_antes = len(df_diario)
df_diario = df_diario.dropna(subset=[col_objetivo, target]).copy()

print("¡Variables de Lag y Promedios Móviles creadas con éxito sin desfase temporal!")
print(f"Se descartaron {filas_antes - len(df_diario)} filas correspondientes a huecos de calendario o sin target al día siguiente.")
print(f"Dimensiones actuales del DataFrame: {df_diario.shape}")

Ahora vmaos a ver los ratios entre consumos de diferentes areas y la eficiencia energetica. Este ratio nos indica lo siguietne:

* Si el valor es menor a 1 significa que la planta o el area consumio menos recursos de los que tenia presupuestados en su meta para este nivel de produccion. Mayor eficiencia.

* Si el valor es exactamente 1 se cumplio la meta al pie de la letra; el consumo real fue identico al planeado

* Si el valor es menor a 1 hubo un sobreconsumo. La planta gastó mas energia, agua o gas del que disctaba la meta, lo cual puede indicar una ineficiencia operativa, una perdida, un desperdicio o un problema en los equipos durante ese dia.

In [ ]:
epsilon = 1e-5
contador_creadas = 0

# Lista de pares exactos (Real, Meta) basados en tus columnas
pares_eficiencia = [
    ('EE Planta / Hl (Consolidado KPI)', 'Meta EE Planta (Consolidado KPI)'),
    ('Agua Planta / Hl (Consolidado KPI)', 'Meta Agua Planta (Consolidado KPI)'),
    ('ET Planta / Hl (Consolidado KPI)', 'Meta ET Planta (Consolidado KPI)'),
    ('Aire Planta / Hl (Consolidado KPI)', 'Meta Aire Planta (Consolidado KPI)'),
    ('EE Elaboracion / Hl (Consolidado KPI)', 'Meta EE Elaboracion (Consolidado KPI)'),
    ('EE Bodega / Hl (Consolidado KPI)', 'Meta EE Bodega (Consolidado KPI)'),
    ('EE Cocina / Hl (Consolidado KPI)', 'Meta EE Cocina (Consolidado KPI)'),
    ('EE Envasado / Hl (Consolidado KPI)', 'Meta EE Envasado (Consolidado KPI)'),
    ('EE Frio / Hl (Consolidado KPI)', 'Meta EE Frio (Consolidado KPI)'),
    ('Agua Elab / Hl (Consolidado KPI)', 'Meta Agua Elab (Consolidado KPI)'),
    ('Agua Bodega / Hl (Consolidado KPI)', 'Meta Agua Bodega (Consolidado KPI)'),
    ('Agua Cocina / Hl (Consolidado KPI)', 'Meta Agua Cocina (Consolidado KPI)'),
    ('Agua Envas / Hl (Consolidado KPI)', 'Meta Agua Envas (Consolidado KPI)')
]

for real_col, meta_col in pares_eficiencia:
    if real_col in df_diario.columns and meta_col in df_diario.columns:
        # Creamos una etiqueta limpia para la nueva columna de eficiencia
        nombre_eficiencia = f"Eficiencia_{real_col.split('/')[0].strip()}"
        df_diario[nombre_eficiencia] = df_diario[real_col] / (df_diario[meta_col] + epsilon)
        contador_creadas += 1

print(f"¡Listo! Se crearon con éxito {contador_creadas} indicadores de eficiencia energética clave.")
print(f"Dimensiones actuales de df_diario: {df_diario.shape}")

* Ratios de Frio vs Produccion de Bodega/Cocina: cuanto frio o energia aociada se requiere por hectolitro producido en bodega (donde la cerveza fermenta y se madura consumiendo mas frio)

* Interacciones con la carga termica: multiplicar los volumenes de mosto caliente o cerveza que ingresan a bodega por los indicadores de frios actuales.

In [ ]:
epsilon = 1e-5

# Ratios específicos orientados a Frío y Bodega
# Relacionamos el consumo de frío con la producción de bodega y de mosto
if 'EE Frio / Hl (Consolidado KPI)' in df_diario.columns:
    
    # Ratio de frío por Hl producido en Bodega
    if 'Hl Producido Bodega (Consolidado Produccion)' in df_diario.columns:
        df_diario['Ratio_Frio_vs_ProdBodega'] = df_diario['EE Frio / Hl (Consolidado KPI)'] / (df_diario['Hl Producido Bodega (Consolidado Produccion)'] + epsilon)
        print("-> Creado: Ratio de Frío vs Producción de Bodega")

    # Ratio de frío por Hl de Mosto (carga térmica inicial)
    if 'Hl de Mosto (Consolidado Produccion)' in df_diario.columns:
        df_diario['Ratio_Frio_vs_Mosto'] = df_diario['EE Frio / Hl (Consolidado KPI)'] / (df_diario['Hl de Mosto (Consolidado Produccion)'] + epsilon)
        print("-> Creado: Ratio de Frío vs Hl de Mosto")


# Interacciones directas 
# Cruzar el consumo actual de frío con los volúmenes de cerveza filtrada o mosto
if 'EE Frio / Hl (Consolidado KPI)' in df_diario.columns:
    
    if 'Hl Cerveza Filtrada (Consolidado Produccion)' in df_diario.columns:
        df_diario['Interaccion_Frio_CervezaFiltrada'] = df_diario['EE Frio / Hl (Consolidado KPI)'] * df_diario['Hl Cerveza Filtrada (Consolidado Produccion)']
        print("-> Creado: Interacción Frío x Cerveza Filtrada")

    if 'Cocimientos Diarios (Consolidado Produccion)' in df_diario.columns:
        df_diario['Interaccion_Frio_Cocimientos'] = df_diario['EE Frio / Hl (Consolidado KPI)'] * df_diario['Cocimientos Diarios (Consolidado Produccion)']
        print("-> Creado: Interacción Frío x Cocimientos Diarios")

print(f"\n¡Variables orientadas al target de Frío creadas con éxito!")
print(f"Dimensiones actuales de df_diario: {df_diario.shape}")

In [ ]:
#  Configurar las coordenadas de Tecate, Baja California, México
# Latitud y Longitud aproximadas de Tecate
lat = 32.5714
lon = -116.6275

# Obtener el rango de fechas de tu dataframe actual
fecha_inicio = df_diario.index.min().strftime('%Y-%m-%d')
fecha_fin = df_diario.index.max().strftime('%Y-%m-%d')

# Consultar la API histórica de Open-Meteo (sin necesidad de instalar librerías extra)
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={fecha_inicio}&end_date={fecha_fin}&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min&timezone=America/Los_Angeles"

print("Descargando datos meteorológicos de Tecate desde Open-Meteo...")
response = requests.get(url)
data = response.json()

# Procesar la respuesta y cargarla en un DataFrame temporal
if "daily" in data:
    df_clima = pd.DataFrame({
        'Fecha': pd.to_datetime(data['daily']['time']),
        'Temp_Media_Tecate': data['daily']['temperature_2m_mean'],
        'Temp_Max_Tecate': data['daily']['temperature_2m_max'],
        'Temp_Min_Tecate': data['daily']['temperature_2m_min']
    })
    df_clima.set_index('Fecha', inplace=True)
    
    df_diario = df_diario.join(df_clima, how='left')
    
    # Rellenar posibles baches vacíos si los hubiera
    df_diario['Temp_Media_Tecate'] = df_diario['Temp_Media_Tecate'].ffill()
    df_diario['Temp_Max_Tecate'] = df_diario['Temp_Max_Tecate'].ffill()
    df_diario['Temp_Min_Tecate'] = df_diario['Temp_Min_Tecate'].ffill()
    
    print(" ¡Temperatura de Tecate integrada con éxito a df_diario!")
else:
    print("⚠️ Hubo un error al consultar la API de Open-Meteo. Verificá la conexión.")

print(f"Dimensiones finales de df_diario con clima: {df_diario.shape}")

## <font color= "orange"> Correcciones pt2

Antes de ver que hacemos con los valores faltantes vamos a analizar la matriz de correlacion para ver si hay variables que son redundantes (aportan la misma informacion) y asi eliminarlas para simplificar el problema.  

In [ ]:
target = 'Target_Frio_Dia_Siguiente'

# --- CORRECCIÓN ANTI-LEAKAGE: Calculamos correlaciones SOLO en el 70% de Train ---
n_train = int(len(df_diario) * 0.70)
df_train_corr = df_diario.iloc[:n_train]

# Tomamos las features numéricas de Train (excluyendo el target)
X_corr = df_train_corr.drop(columns=[target]).select_dtypes(include=[np.number])

# Calculamos la matriz de correlación de Pearson exclusivamente en Train
matriz_corr = X_corr.corr()

umbral = 0.90
correlaciones_altas = []

columns = matriz_corr.columns
for i in range(len(columns)):
    for j in range(i):
        if abs(matriz_corr.iloc[i, j]) > umbral:
            colname_i = columns[i]
            colname_j = columns[j]
            correlaciones_altas.append((colname_i, colname_j, matriz_corr.iloc[i, j]))

df_redundantes = pd.DataFrame(correlaciones_altas, columns=['Variable 1', 'Variable 2', 'Correlacion'])
print(f"Pares de variables con correlación mayor a {umbral} (en Train): {len(df_redundantes)}")
display(df_redundantes.head(15))

matriz_corr_abs = matriz_corr.abs()
mask_alta = (matriz_corr_abs > umbral) & (matriz_corr_abs < 1.0)
columnas_interes = matriz_corr_abs.columns[mask_alta.any()].tolist()

print(f"Variables con correlación >= 90% en Train: {len(columnas_interes)} de {X_corr.shape[1]}")

if len(columnas_interes) > 0:
    matriz_super_filtrada = X_corr[columnas_interes].corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(matriz_super_filtrada, cmap='coolwarm', center=0, annot=False, fmt='.2f', 
                cbar_kws={'label': 'Coeficiente de Correlación de Pearson (Train)'})

    plt.title('Matriz de Correlación (Solo Train) - Clústeres con |r| >= 90%', fontsize=14, pad=15)
    plt.xlabel('Variables', fontsize=12)
    plt.ylabel('Variables', fontsize=12)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron variables con una correlación del 90% o más.")

Sin embargo vamos a hacer un filtro donde verificamos que no se estén eliminando variables que nos ayuden a predecir nuestra variable objetivo.  Utilizamos el coeficiente de Pearson para calcular un p-value. 

Hipotesis Nula: No hay correlacion lineal entre la variable y el target (correlacion=0).

Hipoteisis Alternativa: Si hay una correlacion real 

Si el p-value<0.05 rechazamos la hipotesis nula. La relacion es estadisticamente significativa, por lo que la variable se queda.

Si el p-value>=0.05 no podemos rechazar la nulidad, la aparente correlacion puede ser pura casualisas y esa variable se puede eliminar porque no aporta informacion confiable.

In [ ]:
warnings.filterwarnings('ignore', category=UserWarning)

target = 'Target_Frio_Dia_Siguiente'

# --- CORRECCIÓN ANTI-LEAKAGE: Recorte estricto al 70% de Entrenamiento ---
n_train = int(len(df_diario) * 0.70)
df_train_only = df_diario.iloc[:n_train]

# 1. Eliminamos variables repetidas entre sí (|r| >= 0.90) evaluando SOLO en Train
X_features_train = df_train_only.drop(columns=[target]).select_dtypes(include=[np.number])
matriz_corr = X_features_train.corr().abs()

upper_tri = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))
umbral_redundancia = 0.90

cols_a_eliminar_redundancia = set()
for col in upper_tri.columns:
    correlacionadas = upper_tri.index[upper_tri[col] >= umbral_redundancia].tolist()
    if correlacionadas:
        grupo = correlacionadas + [col]
        # De las redundantes, elegimos la que tenga mayor correlación con el target EN TRAIN
        mejor_var = df_train_only[grupo].corrwith(df_train_only[target]).abs().idxmax()
        for c in grupo:
            if c != mejor_var:
                cols_a_eliminar_redundancia.add(c)

# Aplicamos la eliminación de columnas redundantes a todo el dataset
df_sin_redundancia = df_diario.drop(columns=list(cols_a_eliminar_redundancia))
print(f"Paso 1 - Redundancia (calculada en Train): Se eliminaron {len(cols_a_eliminar_redundancia)} variables repetidas.")

# 2. PASO DE SIGNIFICANCIA ESTADÍSTICA (P-VALUE): Filtrar ruido con p < 0.05 SOLO en Train
df_train_sin_red = df_sin_redundancia.iloc[:n_train]
X_final_candidatas = df_train_sin_red.drop(columns=[target]).select_dtypes(include=[np.number])
resultados_pv = []

for col in X_final_candidatas.columns:
    datos_validos = df_train_sin_red[[col, target]].dropna()
    if len(datos_validos) > 2 and datos_validos[col].nunique() > 1:
        corr, p_value = pearsonr(datos_validos[col], datos_validos[target])
        resultados_pv.append({'Variable': col, 'P_Value': p_value})

df_pv = pd.DataFrame(resultados_pv)
umbral_significancia = 0.05
variables_significativas = df_pv[df_pv['P_Value'] < umbral_significancia]['Variable'].tolist()

# Filtramos el dataset completo usando ÚNICAMENTE la lista de variables aprendida en Train
df_diario = df_sin_redundancia[variables_significativas + [target]]

print(f"Paso 2 - P-Value (calculado en Train): De las restantes, {len(variables_significativas)} pasaron el test (p < {umbral_significancia}).")
print(f"Dimensiones finales del dataset libre de Data Leakage: {df_diario.shape}")

## <font color= "orange"> Manejo de Valores Faltantes

In [ ]:
nulos_por_columna = df_diario.isna().sum()
# print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False))
print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False).to_string())

In [ ]:
#Graficamos la matriz de nulidad para ver dónde están parados los huecos
plt.figure(figsize=(14, 6))
msno.matrix(df_diario, fontsize=10, color=(0.2, 0.4, 0.6))
plt.title("Matriz de Nulidad - Dataset de Cervecería", fontsize=14, fontweight='bold')
plt.show()

Podemos ver a través de la matriz de nulidad que hay columnas totalmente vacias. Inicialmente a estas columnas debemos eliminarlas por completo porque ensucia y arruina cualquier calculo o modelo posterior.

In [ ]:
print(f" Cantidad de columnas después de limpiar las vacías: {df_diario.shape[1]}")

#Eliminamos todas las columnas que esten 100% vacías
df_diario=df_diario.dropna(axis=1, how='all')

print(f" Cantidad de columnas después de limpiar las vacías: {df_diario.shape[1]}")

In [ ]:
nulos_por_columna = df_diario.isna().sum()
print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False).to_string())

Podemos ver que hay features que son "unnamed" que solo tienen 1 o 2 datos. A estas las vamos a eliminar ya que primero tienen mucha cantidad de nulos y ademas no se sabe siqueira que es lo que mide, lo cual tampoco nos serviria para nuestra variable objetivo a predecir. 


In [ ]:
# Filtramos el DataFrame conservando únicamente las columnas que NO tengan 'Unnamed' en su nombre
df_diario=df_diario.loc[:, ~df_diario.columns.str.contains('Unnamed', case=False, na=False)]

nulos_por_columna = df_diario.isna().sum()
print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False).to_string())

Recordemos ademas que nustra variable a predecir es el Frio (Kw), asi que aquellas que tengan muchisimos nulos y encima no nos ayuda a predecir las eliminamos. Para esto vamos a analizar las variables que tenemos.

1. Metas (Meta EE, Meta Agua..):
No nos sirven para predecir ya que las metas son objetivos teoricos en planillas creados por gestion corporativa, no mediciones reales del proceso fisico

2. Identificadores y Auxiliares (Id(...)):
No nos sirven ya que son claves primarias autogeneradas o codigos de filas de la base de datos original. Tampoco representa ninguna magnitud física.

4. Consolidado EE:
Esta SI nos sirve. Planta(Kw), Bodega (Kw), Elaboracion(Kw), etc, estan directamente correlacionadas con el consumo de frio. El sistema de refrigeracion enfría principalmente los tanques de fermentacion y reposo (Bodega/Elaboracion)

5. Totalizadores Energía (Km Mycom 1-7, KW Trafo):
Esta SI nos sirve. Los compresores Mycom son los motores del sistema de refrigeracion por amoniaco/glicol que generan el frio industrial. La suma de la potencia de los Mycom equivale casi al 100% de Frio(Kw)

6. 7. Consolidado Produccion y Totalizadores Produccion (Hl Producido, Hl Mosto):
Estas SI nos sirven. Enfriar el mosto caliente que sale de la cocina hasta la temperatura de fermentacion y mantener los hectolitos en tanques de reposo a 0°C es lo principal del consumo electrico de frío

8. Totalizadores Glicol (Tot Reposo, Tot Fermentacion):
Estas SI nos servirían. El glicol es el fluido refrigerante que transporta el frio a los tanques. Sus mediciones explican directamente la carga de frio.

9. Agua, GasVapor, Aire y CO2:
Indican el nivel general de actividad de la planta (si la planta envasa o elabora, consume vapor, aire y augua). Sin embargo, aportan indirectamente al frio. Si tienen mas de 15 mil nulos es preferible descartarlas frente a los datos directos de energia y produccion.

In [ ]:
#Calculamos el porcentaje de nulos por columna y ordenarlamos de mayor a menor
porcentaje_nulos = (df_diario.isna().mean() * 100).sort_values(ascending=False)
print(porcentaje_nulos[porcentaje_nulos > 0].to_string())

Por estos motivos eliminamos no solo aquellas variables que no sirven sino que ademas eliminamos aquellas que tienen mucha cantidad de nulos.

In [ ]:
#Definimos qué patrones queremos eliminar por completo del DataFrame
patrones_a_eliminar=[
    'Meta',          #Objetivos teóricos y planillas de gestión
    'Id ',           #Identificadores internos
    'Auxiliar',      #Columnas auxiliares vacías o de soporte
    'Ultimo Dato',   #Seguimiento diario suelto
]

#Creamos una condición para filtrar las columnas que NO contengan estos patrones
columnas_a_conservar = []
for col in df_diario.columns:
    #Verificamos si la columna contiene alguno de los patrones prohibidos
    contiene_prohibido = any(patron.lower() in str(col).lower() for patron in patrones_a_eliminar)
    
    #Conservamos la columna solo si no tiene nada prohibido (y mantenemos DIA/HORA)
    if not contiene_prohibido or col in ['DIA', 'HORA']:
        columnas_a_conservar.append(col)

#Aplicamos el filtro al DataFrame
df_diario = df_diario[columnas_a_conservar]

print(f" Cantidad de columnas limpias y útiles conservadas: {df_diario.shape[1]}")
print("\n Primeras 20 columnas resultantes para validar:")
print(df_diario.columns[:20].tolist())

In [ ]:
nulos_por_columna = df_diario.isna().sum()
print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False).to_string())

In [ ]:
#Graficamos la matriz de nulidad para ver dónde están parados los huecos
plt.figure(figsize=(14, 6))
msno.matrix(df_diario, fontsize=10, color=(0.2, 0.4, 0.6))
plt.title("Matriz de Nulidad - Dataset de Cervecería", fontsize=14, fontweight='bold')
plt.show()

Ahora tenemos que ver como tratar estas variables que tienen muchos nulos. No las podemos eliminar, porque como dijimos antes estas sirven para predecir nuerstra variable objetivo. 

Para saber que hacer tenemos que saber si es MNAR o MAR:
* MCAR: Significa que lso datos faltan por puro azar, de forma totalmente independiente de cualquier cosa. Seria como si el sistema de almacenamiento hubiese tirado una moneda y borrado valores al azar por simple mala suerte o un bug informatico. 
Si una variable tiene 15mil o 19mil casilleros vacios no es por azar. Es porque el equipo no existia todavia, estuvo apagado por baja demanda, o el sensor se rompio y dejo de transmitir por meses. Esto no es aleatorio, asi que descartamos que sea MCAR.

* MAR: El valor falta no por culpa de la variable en si misma, sino por otra variable que si estamos midiendo en el dataset. En nuestro caso, imaginemos que un sensor secundario de una linea de envasado deja de registrar datos con mas frecuencia los findes de semana (porque hay menos personal operativo y se desenergizan ciertos subsistemas auxiliares). El faltante dependeria de la variable Dia de la semana no de la lectura de la maquina.

* MNAR: El dato falta precisamente por el valor que deberia haber tenido. Por ejemplo un sensor de temperatura deja de funcionar porque la planta sufrio una falla critica o una sobrecarga de temperatura que rompio el instrumento. Si ignoramos el nulo o lo interpolamos a ciegas estariamos ocultando una catastrofe operativa.


Los **faltantes masivos (790 o 600 datos faltantes) corresponden a MNAR**. Si una variable le falta el 60% o 70% no es al azar ni depende de otra variable. Esto suele pasar porque el sensor no estaba instalado todavia, la linea de produccion estuvo parada durante meses, o el equipo se dio de baja y dejo de reportar al sistema. 

Inventar tanta cantidad de datos de un sensor es un error porque estariamos inventando la historia de la planta. Lo mas correcto para nosotros es descartar esas variables por completo del modelo o, asumir que esos periodos de ausencia significan que la planta estuvo detenida.

Los **faltantes intermedios (300 o  400 datos faltantes) corresponden a MAR**. Cuando se corta el agua o se apaga un sector, caen en bloque varias variables al mismo tiempo. 

Su ausencia depende de variables operativas observables. En este caso aplican imputadores como KNN o MICE.

Los **faltantes minimos (1 o 22 datos faltantes) corresponden a MCAR**. Esto podria ser un corte de luz puntual de un dia, un reinicio del servidor o un error de escritura en la base de datos de esa jornada especifica. 

Se puede resolver mediante media o mediana. 

In [ ]:
#Calculamos el porcentaje de nulos por columna y ordenarlamos de mayor a menor
porcentaje_nulos = (df_diario.isna().mean() * 100).sort_values(ascending=False)
print(porcentaje_nulos[porcentaje_nulos > 0].to_string())

Realizamos la particion en entrenamiento y prueba de manera previa a cualquier imputacion. Las reglas de imputacion (como KNN o imputacion por la media/mediana) se calculan estrictamente sobre el conjunto de entrenamiento y se aplicaon como una proyeccion sobre el conjunto de prueba, garantizando que no existiera ningun tipo de data leakage.

Hay que tener cuidado porque no podemos inventar un numnero para el periodo en el que un sensor ni siquiera existia ya que arruinaria por completo la integridad de los datos y sesgaria el modelo. Tampoco podemos eliminarlos porque nos pueden ayudar a predcir nuestra variable objetivo. Aquellos que cumplan con eso los dejaremos como NaN

In [ ]:
sensores_incorporados_tarde = []

# Evaluamos exclusivamente sobre el tramo de entrenamiento (70% inicial)
n_train = int(len(df_diario) * 0.70)
df_train_check = df_diario.iloc[:n_train]

inicio_corte = int(n_train * 0.30) # Primer 30% de Train
fin_corte = int(n_train * 0.70)    # Último 30% de Train

for col in df_train_check.columns:
    if col == 'Target_Frio_Dia_Siguiente':
        continue
    serie = df_train_check[col]
    
    nulos_inicio = serie.iloc[:inicio_corte].isnull().mean()
    nulos_fin = serie.iloc[fin_corte:].isnull().mean()
    
    if nulos_inicio > 0.50 and nulos_fin < 0.20:
        sensores_incorporados_tarde.append(col)

print(f"--- SENSORES INCORPORADOS TARDE IDENTIFICADOS EN TRAIN ({len(sensores_incorporados_tarde)}) ---")
print(sensores_incorporados_tarde)

In [ ]:
target = 'Target_Frio_Dia_Siguiente'
X = df_diario.drop(columns=[target])
y = df_diario[target]

#DIVISIÓN CRONOLÓGICA (Train 70% / Val 15% / Test 15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)
val_prop = 0.15 / 0.85 
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_prop, shuffle=False)

#DEFINIR EL VECTOR DE SENSORES PROTEGIDOS (Incorporados tarde)
#Estos son los 12 sensores que detectamos que arrancaron tarde y cuyos NaN NO deben tocarse
sensores_protegidos = sensores_incorporados_tarde
#SEPARAMOS EL DATAFRAME EN DOS MUNDOS PARA EL CONTROL DE NULOS:
# Las variables normales (sujetas a MCAR/MAR/MNAR e imputación)
#Las variables protegidas (cuyos NaN se quedan intactos)
cols_normales = [col for col in X_train.columns if col not in sensores_protegidos]

X_train_norm = X_train[cols_normales]
X_train_prot = X_train[sensores_protegidos]

X_val_norm = X_val[cols_normales]
X_val_prot = X_val[sensores_protegidos]

X_test_norm = X_test[cols_normales]
X_test_prot = X_test[sensores_protegidos]


#IDENTIFICACIÓN DE NULOS Y CLASIFICACIÓN (Solo sobre las variables normales)
porcentaje_nulos = (X_train_norm.isnull().sum() / len(X_train_norm)) * 100

cols_mcar = porcentaje_nulos[(porcentaje_nulos > 0) & (porcentaje_nulos < 1.0)].index.tolist()
cols_mar = porcentaje_nulos[(porcentaje_nulos >= 1.0) & (porcentaje_nulos <= 45.0)].index.tolist()
cols_mnar = porcentaje_nulos[porcentaje_nulos > 45.0].index.tolist()

# Eliminamos las columnas MNAR solo del bloque normal
X_train_norm = X_train_norm.drop(columns=cols_mnar, errors='ignore')
X_val_norm = X_val_norm.drop(columns=cols_mnar, errors='ignore')
X_test_norm = X_test_norm.drop(columns=cols_mnar, errors='ignore')


#TRATAMIENTO MCAR (< 1%) -> SimpleImputer (Media)
cols_mcar_activas = [c for c in cols_mcar if c in X_train_norm.columns]
if len(cols_mcar_activas) > 0:
    imputer_mcar = SimpleImputer(strategy='mean')
    X_train_norm[cols_mcar_activas] = imputer_mcar.fit_transform(X_train_norm[cols_mcar_activas])
    X_val_norm[cols_mcar_activas] = imputer_mcar.transform(X_val_norm[cols_mcar_activas])
    X_test_norm[cols_mcar_activas] = imputer_mcar.transform(X_test_norm[cols_mcar_activas])


# TRATAMIENTO MAR (1% - 45%) -> KNNImputer
cols_mar_activas = [c for c in cols_mar if c in X_train_norm.columns]
if len(cols_mar_activas) > 0:
    imputer_mar = KNNImputer(n_neighbors=5)
    X_train_norm[cols_mar_activas] = imputer_mar.fit_transform(X_train_norm[cols_mar_activas])
    X_val_norm[cols_mar_activas] = imputer_mar.transform(X_val_norm[cols_mar_activas])
    X_test_norm[cols_mar_activas] = imputer_mar.transform(X_test_norm[cols_mar_activas])


#RECONSTRUIMOS LOS CONJUNTOS FINALES UNIENDO LO NORMAL (IMPUTADO) + LO PROTEGIDO (CON SUS NANDS INTACTOS)
X_train = pd.concat([X_train_norm, X_train_prot], axis=1)
X_val = pd.concat([X_val_norm, X_val_prot], axis=1)
X_test = pd.concat([X_test_norm, X_test_prot], axis=1)

print("\n¡Preprocesamiento y protección de sensores tardíos completados sin fuga de datos!")
print(f"Dimensiones finales de X_train: {X_train.shape}")
print(f"Dimensiones finales de X_val:   {X_val.shape}")
print(f"Dimensiones finales de X_test:  {X_test.shape}")

Ahora unificamos klas variables en el dataframe para poder continuar con nuestro analisis

In [ ]:
# unificamos las variables predictoras (features) ya imputadas

X_imputado = pd.concat([X_train, X_val, X_test])
y_unificado = pd.concat([y_train, y_val, y_test])

df_diario = X_imputado.copy()
df_diario[target] = y_unificado
df_diario = df_diario.sort_index()

print(f"Dimensiones finales del DataFrame unificado: {df_diario.shape}")
print(f"¿Quedan nulos?: {df_diario.isnull().sum().sum()}")

In [ ]:
#Graficamos la matriz de nulidad para ver dónde están parados los huecos
plt.figure(figsize=(14, 6))
msno.matrix(X_train, fontsize=10, color=(0.2, 0.4, 0.6))
plt.title("Matriz de Nulidad - Dataset de Cervecería", fontsize=14, fontweight='bold')
plt.show()

Esta es la confirmacion visual de que hemos tratado a los valores nulos, pero protegiendo los sensores que se incorporaron tarde. Podemos ver que en la parte superior (2020) esas columnas tienen espacios en blanco porque el sensor aun no habia sido instalado; y a medida que bajamos en el tiempo (hacia el presente), se vuelven de color azul porque empezaron a registrar de forma estable.

# <font color= "orange"> Eliminacion o Correccion de Outliers extremos


Primero vamos a generar boxplots para ver a grandes rasgos que variables tienen valores atipicos. 

In [ ]:
#Seleccionamos todas las columnas numéricas del DataFrame unificado
columnas_numericas = df_diario.select_dtypes(include=[np.number]).columns.tolist()

# Configuramos la visualización en subplots (por ejemplo, bloques de 6 columnas por figura)
# Esto evita que los nombres de los sensores se encimen y no se entiendan.
n_cols = 4  # Cantidad de columnas en la grilla de gráficos
n_vars = len(columnas_numericas)
n_rows = (n_vars + n_cols - 1) // n_cols

plt.figure(figsize=(16, n_rows * 3))

for i, col in enumerate(columnas_numericas):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.boxplot(y=df_diario[col], color='skyblue')
    plt.title(col, fontsize=9)
    plt.ylabel('')
    plt.xlabel('')

plt.tight_layout()
plt.suptitle("Boxplots de Variables - Control de Outliers", fontsize=14, y=1.02)
plt.show()

Para la deteccion de outliers univariados vamos a usar Puntuaciones z Modificadas (con MAD): Es la mejor para series temporales industriales porque usa la Mediana y la MAD en lugar de la media y la desviacion estandar clasica. Es robusta ante los extremos.

Como metodo multivariado usamos LOF. Es muy importante porque imaginemos que un sensor de temperatura marca un valor perfectamente normal por si solo y el caudal de agua tambien. Pero juntos (alta temperatura con cero caudal) son fisicamente imposibles y representan un error o una anomalia grave. El LOF detecta esos puntos que son atipicos en el espacio multidimensional de la planta.

In [ ]:
#TRATAMIENTO DE OUTLIERS (Z-Modificada + Clipping) SIN FUGA DE DATOS

def aplicar_limpieza_outliers(df_train, df_val, df_test, umbral_z=3.5, lower_q=0.01, upper_q=0.99):
    
    #Hacemos copias para no pisar los originales
    tr = df_train.copy()
    va = df_val.copy()
    te = df_test.copy()
    
    # Seleccionamos las columnas numéricas (excluyendo el target)
    cols_numericas = [col for col in tr.select_dtypes(include=[np.number]).columns if col != 'Target_Frio_Dia_Siguiente']
    
    for col in cols_numericas:
        #Z-MODIFICADA (Cálculo de parámetros SOLO en Train)
        mediana = tr[col].median()
        mad = np.median(np.abs(tr[col] - mediana))
        
        if mad > 0:
            # Límites basados en la mediana y MAD del TRAIN
            lim_inf_z = mediana - (umbral_z * mad / 0.6745)
            lim_sup_z = mediana + (umbral_z * mad / 0.6745)
            
            # Aplicamos el acotamiento usando los límites de Train en los tres conjuntos
            tr[col] = tr[col].clip(lower=lim_inf_z, upper=lim_sup_z)
            va[col] = va[col].clip(lower=lim_inf_z, upper=lim_sup_z)
            te[col] = te[col].clip(lower=lim_inf_z, upper=lim_sup_z)
            
        #WINSORIZACIÓN
        p_inf = tr[col].quantile(lower_q)
        p_sup = tr[col].quantile(upper_q)
        
        # Aplicamos los percentiles de Train en los tres conjuntos
        tr[col] = tr[col].clip(lower=p_inf, upper=p_sup)
        va[col] = va[col].clip(lower=p_inf, upper=p_sup)
        te[col] = te[col].clip(lower=p_inf, upper=p_sup)
        
    return tr, va, te

# Ejecutamos la función pasándole los tres conjuntos ya divididos e imputados previamente
X_train_limpio, X_val_limpio, X_test_limpio = aplicar_limpieza_outliers(X_train, X_val, X_test)

print("¡Tratamiento de outliers aplicado con éxito sin fuga de datos!")
print(f"Dimensiones finales - Train: {X_train_limpio.shape} | Val: {X_val_limpio.shape} | Test: {X_test_limpio.shape}")

In [ ]:
#Si querés graficar cómo quedó todo el dataset procesado (juntando las particiones limpias solo para visualizar)
df_total_limpio = pd.concat([X_train_limpio, X_val_limpio, X_test_limpio])

# O si preferís graficar exclusivamente el train limpio:
# df_a_graficar = X_train_limpio

# Seleccionamos las columnas numéricas (excluyendo el target si estuviera ahí)
cols_a_graficarlas = [col for col in df_total_limpio.select_dtypes(include=[np.number]).columns if col != 'Target_Frio_Dia_Siguiente']

n_cols = 4  
n_vars = len(cols_a_graficarlas)
n_rows = (n_vars + n_cols - 1) // n_cols

plt.figure(figsize=(16, n_rows * 3))

for i, col in enumerate(cols_a_graficarlas):
    plt.subplot(n_rows, n_cols, i + 1)
    # Graficamos el boxplot del dataset procesado
    sns.boxplot(y=df_total_limpio[col], color='lightgreen')
    plt.title(f'Post-Limpieza: {col}', fontsize=9, fontweight='bold')
    plt.ylabel('')
    plt.xlabel('')

plt.tight_layout()
plt.suptitle("Boxplots de Variables - Después del Tratamiento de Outliers y Nulos", fontsize=14, y=1.02)
plt.show()

Comparando los boxplots podemos ver que antes, los valores maximos de variables como Agua Elab o ET Elab se iban a 100,400 0 mas de 1000 debido a errores drasticos de sensores o picos absurdos. Despues de haber tratado los valores atipicos podemos ver como estas escalas se achicaron drasticamente (por ejemplo Agua Elab ahora corta cerca de 3 y ET Elab en 50). Esto demuestra que el Clipping acotó exitosamente los vaores extremos e imposibles.

Podemos ver que siguen apareciendo atipicos, sin embargo es porque el boxplot calcula sus bigotes en funcion de la distribucion actual de ls datos. Al comprimir los datos extremos con el clipping, el rango intercuartilico (IQR) se recalcula y e ajusta a la nueva dispresion. Por lo tanto, los puntos quedan un poco mas alejado del nuevo cuerpo central siendo marcados como outliers por el grafico.

En una cerveceria el objetivo no es eliminar toda la variabilidad de la planta hasta que quede una linea recta perfecta, sino eliminar los errores grotescos (como leer 3000 cuando el maximo real es 50). 

Los puntos que vemos ahora son la variacion natural del proceso productivo.

En los boxplot tambien se puede ver que hay variables que aparentan estar un poco sesgadas (asimetria). Cuando apliquemos modelos lineales como Ridge o Lasso es fundamental y obligatorio  hacer un tratamiento de esto. Los modelos lineales asumen relaciones lineales y distribuciones cercanas a la normalidad. Si las features o el Target tienen una asimetria brutal, el modelo se desbalancea intentando cazar los valores extremos.

Primero calculamos el skewness:
* Si la asimetria es baja (<1) no hace falta tocar nada.
* Si la asimetria es muy alta (>1): deberiamos tratarlos con transformaciones antes de entrenar los modelos.

In [ ]:
X_train_num = X_train_limpio.select_dtypes(include=['number'])

#calculamos el skewness exclusivamente en X_train
skew_values = X_train_num.skew().sort_values(ascending=False)

# Creamos un DataFrame ordenado para ver cuáles están más sesgadas
df_skew = pd.DataFrame({'Skewness': skew_values})
print("TOP 10 VARIABLES MÁS SESGADAS (EN X_TRAIN)")
print(df_skew.head(10))

In [ ]:
numeric_cols = X_train_limpio.select_dtypes(include=[np.number]).columns

#Calculamos el skewness SOLO en X_train_norm para ver cuáles superan el umbral (ej. |skew| > 1.0)
skewness = X_train_limpio[numeric_cols].skew()
columnas_sesgadas = skewness[abs(skewness) > 1.0].index.tolist()

print(f"Variables altamente sesgadas detectadas en Train: {len(columnas_sesgadas)}")

Para solucionar el problema dela asimetria vamos a aplicar transformaciones. Aquellas transformaciones que aprenden de los parametros de los datos (como box-cox o Yeo-Jhonson) deben ajustarse unicamente con el conjunto de entrenamiento.

In [ ]:
X_train_scaled = X_train_limpio.copy()
X_val_scaled   = X_val_limpio.copy()
X_test_scaled  = X_test_limpio.copy()

X_train_num = X_train_limpio.select_dtypes(include=[np.number])
#  Calculamos el skewness en Train y filtramos las que tengan asimetría absoluta > 1.0
skew_vals = X_train_num.skew()
cols_sesgadas = skew_vals[abs(skew_vals) > 1.0].index.tolist()


print(f"Variables con asimetría |skew| > 1.0 detectadas en Train: {len(cols_sesgadas)}")

if len(cols_sesgadas) > 0:
    # Guardamos una copia del ANTES para comparar
    X_train_antes = X_train_limpio[cols_sesgadas].copy()
    
    # Aplicamos el PowerTransformer (Yeo-Johnson) ajustado SOLO en Train
    transformer = PowerTransformer(method='yeo-johnson', standardize=False)
    X_train_scaled[cols_sesgadas] = transformer.fit_transform(X_train_scaled[cols_sesgadas])
    X_val_scaled[cols_sesgadas]   = transformer.transform(X_val_scaled[cols_sesgadas])
    X_test_scaled[cols_sesgadas]  = transformer.transform(X_test_scaled[cols_sesgadas])
    # Guardamos el DESPUÉS
    X_train_despues = X_train_scaled[cols_sesgadas].copy()
    
    # Graficamos el ANTES vs DESPUÉS (mostramos hasta 4 variables de ejemplo para mantener la claridad visual)
    n_vars_a_graficar = min(27, len(cols_sesgadas))
    
    fig, axes = plt.subplots(n_vars_a_graficar, 2, figsize=(14, 4 * n_vars_a_graficar))
    if n_vars_a_graficar == 1:
        axes = np.array([axes])
        
    for i, col in enumerate(cols_sesgadas[:n_vars_a_graficar]):
        # Gráfico del ANTES
        sns.histplot(X_train_antes[col], kde=True, ax=axes[i, 0], color='salmon', bins=30)
        axes[i, 0].set_title(f"ANTES: {col}\nSkewness: {skew_vals[col]:.2f}", fontsize=10, fontweight='bold')
        axes[i, 0].set_xlabel("")
        
        # Gráfico del DESPUÉS
        nuevo_skew = X_train_despues[col].skew()
        sns.histplot(X_train_despues[col], kde=True, ax=axes[i, 1], color='seagreen', bins=30)
        axes[i, 1].set_title(f"DESPUÉS (Yeo-Johnson): {col}\nSkewness: {nuevo_skew:.2f}", fontsize=10, fontweight='bold')
        axes[i, 1].set_xlabel("")
        
    plt.suptitle("Comparativa de Asimetría: Antes vs. Después de la Transformación", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron variables con un skewness absoluto mayor a 1.0.")

Podemos ver que en su gran mayoria pudo acomodar la asimetria llevandolo a una distribucion casi normal. Ademas fueron estandarizadas, lo cual nos ayudara en el futuro.

Por otro ladopodemos ver que hay variables que tienen una distribucion extremadamente sesgada que ninugna funcion estandar (ni Box-Cox ni Yeo-Jhonson) puede normalizar por completo. En plantas industriales esto pasa mucho con contadores o totalizadores que pasan gran parte del tiempo en ceros o inactivos.

In [ ]:
# Seleccionamos las columnas numéricas que van a ser estandarizadas 
# (excluyendo el target y los sensores protegidos que dejamos con sus NaN)
cols_a_estandarizar = X_train_scaled.select_dtypes(include=[np.number]).columns

if len(cols_a_estandarizar) > 0:
    scaler = StandardScaler()
    X_train_scaled[cols_a_estandarizar] = scaler.fit_transform(X_train_scaled[cols_a_estandarizar])
    X_val_scaled[cols_a_estandarizar]   = scaler.transform(X_val_scaled[cols_a_estandarizar])
    X_test_scaled[cols_a_estandarizar]  = scaler.transform(X_test_scaled[cols_a_estandarizar])

    print("¡Estandarización completada con éxito")
    print(f"Media de control en X_train (debería ser ~0): {X_train_scaled[cols_a_estandarizar].mean().mean():.4f}")
    print(f"Desviación estándar de control en X_train (debería ser ~1): {X_train_scaled[cols_a_estandarizar].std().mean():.4f}")
else:
    print("No se encontraron variables numéricas para estandarizar.")

# <font color= "red"> Seleccion de Variables

Teniendo mas de 400 features despues de todo el preprocesamiento, aplicar una seleccion rigurosa es fundamental para evitar el overfitting y dejar al modelo unicamente con las señales que realmente explican la demanda de frio.

Comparamos tres metodos en `notebooks/seleccion_variables.ipynb`: **SelectKBest con ANOVA F** (`f_regression`, filtro univariado), **RFE con Random Forest** (envoltura) y **LASSO** (embebido, penalizacion L1). Cada uno se ajusto solo con Train y se evaluo con `TimeSeriesSplit` dentro de Train y luego en Validation, con Ridge y LightGBM, para k entre 5 y 80 variables. Test no se uso.

Nos quedamos con **SelectKBest (ANOVA F) y k = 10** por cuatro criterios:

1. **Menor MAE en la validacion cruzada temporal**: 3.323 kW con k = 10 (3.248 con k = 5), frente a 3.351 de RFE y 3.542 de LASSO en su mejor k.
2. **Empate en Validation**: 2.273 kW contra 2.220–2.241 de RFE, una diferencia menor que la variacion entre folds; LASSO queda en 2.467.
3. **Mayor estabilidad**: repite el 62 % de sus 10 variables entre folds (Jaccard), contra 41 % de RFE y 26 % de LASSO.
4. **Coherencia con el EDA**: elige el Frio del dia con sus rezagos y media movil y los consumos de planta mas correlacionados con el target.

Con estas 10 variables, Ridge baja el MAE de Validation a 2.273 kW, un 8,5 % mejor que el mejor baseline (media movil de 7 dias, 2.484 kW). Todavia no llega al 10 % de mejora: el proximo paso es volver a optimizar los hiperparametros con el conjunto reducido.

Algunos sensores se instalaron tarde y conservan faltantes despues de escalar. Para calcular el estadistico F los completamos con la mediana de Train y acotamos los valores escalados a ±8 desvios, igual que en la comparacion de metodos. Esa imputacion se usa solo para ordenar las variables: el dataset exportado mantiene los valores originales.

In [ ]:
from sklearn.feature_selection import f_regression

K_VARIABLES = 10  # justificado en notebooks/seleccion_variables.ipynb

# Faltantes de sensores tardios -> mediana de Train; valores escalados acotados a ±8 desvios
X_sel = X_train_scaled.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
X_sel = X_sel.fillna(X_sel.median()).fillna(0).clip(-8, 8)

# SelectKBest con ANOVA F: estadistico F de cada variable contra el target, calculado solo con Train
puntaje_f, p_valor = f_regression(X_sel, y_train.loc[X_sel.index])
puntajes_f = (pd.DataFrame({'F': np.nan_to_num(puntaje_f), 'p_valor': p_valor}, index=X_sel.columns)
              .sort_values('F', ascending=False))
features_finales = puntajes_f.index[:K_VARIABLES].tolist()

print(" REPORTE DE SELECCIÓN DE VARIABLES")
print(f"• Variables candidatas: {len(puntajes_f)}")
print(f"• Seleccionadas por SelectKBest (ANOVA F): {len(features_finales)}")
display(puntajes_f.head(K_VARIABLES).round({'F': 0}))

# Reducimos los sets definitivos para el modelado
X_train_final = X_train_scaled[features_finales]
X_val_final   = X_val_scaled[features_finales]
X_test_final  = X_test_scaled[features_finales]

print(f"\n• Shape final X_train: {X_train_final.shape}")
print(f"• Shape final X_val:   {X_val_final.shape}")
print(f"• Shape final X_test:  {X_test_final.shape}")

fig, ax = plt.subplots(figsize=(10, 6))
top_f = puntajes_f['F'].head(20).iloc[::-1]
colores = ['royalblue' if variable in features_finales else 'lightgray' for variable in top_f.index]
top_f.plot(kind='barh', ax=ax, color=colores)
ax.set_title(f'Top 20 variables por estadístico F (en azul, las {K_VARIABLES} seleccionadas)')
ax.set_xlabel('F (ANOVA, f_regression)')
plt.tight_layout()
plt.show()

El conjunto final queda formado por las 10 variables con mayor estadistico F en Train. Son señales con relacion fisica directa con la demanda de frio: el consumo de Frio del dia, su media movil de 7 dias y sus rezagos de 1 y 7 dias, y consumos de la planta muy correlacionados con el target, como el Trafo 11, el agua de planta y los condensadores 5, 6 y 9. Como el ANOVA F evalua cada variable por separado, puede elegir variables correlacionadas entre si; el filtro previo de redundancia (|r| >= 0.90) lo mitiga. La comparacion completa con RFE y LASSO, con sus tablas y graficos, esta en `notebooks/seleccion_variables.ipynb`.

Crea un diccionario con texto plano detallando el script, la fecha actual y las transformaciones que fuimos haciendo.

Toma las dimensiones reales del dataframe para que quede registrado tecnicamente.

Lo serializa y lo guarda en formato JSON

In [ ]:
# Usamos X_train_limpio, X_val_limpio y X_test_limpio (depurados pero SIN ESCALAR, como exige el punto 2.4)
train_full_sin_escalar = X_train_limpio[features_finales].copy()
train_full_sin_escalar['Target_Frio_Dia_Siguiente'] = y_train
train_full_sin_escalar['particion'] = 'train'

val_full_sin_escalar = X_val_limpio[features_finales].copy()
val_full_sin_escalar['Target_Frio_Dia_Siguiente'] = y_val
val_full_sin_escalar['particion'] = 'val'

test_full_sin_escalar = X_test_limpio[features_finales].copy()
test_full_sin_escalar['Target_Frio_Dia_Siguiente'] = y_test
test_full_sin_escalar['particion'] = 'test'

# Concatenar todo en una sola tabla maestra y ordenar cronológicamente por el índice DIA
df_dataset_final = pd.concat([
    train_full_sin_escalar, 
    val_full_sin_escalar, 
    test_full_sin_escalar
]).sort_index()

# Verificamos que cada etiqueta coincida con el cierre de Frío de D + 1.
dia_objetivo = df_dataset_final.index + pd.Timedelta(days=1)
frio_esperado = frio_calendario.reindex(dia_objetivo).to_numpy()
etiquetas_validas = df_dataset_final['Target_Frio_Dia_Siguiente'].eq(frio_esperado)
if not etiquetas_validas.all():
    ejemplos = dia_objetivo[~etiquetas_validas.to_numpy()][:5].strftime('%Y-%m-%d').tolist()
    raise ValueError(f'Hay etiquetas que no coinciden con el cierre de D+1: {ejemplos}.')

carpeta_procesados = raiz_proyecto / 'data' / 'processed'
carpeta_procesados.mkdir(parents=True, exist_ok=True)
ruta_csv_fisica = carpeta_procesados / 'dataset_final.csv'
ruta_csv_key = 'data/processed/dataset_final.csv'

df_dataset_final.to_csv(ruta_csv_fisica, index=True)
print(f"¡Dataset final (limpio, filtrado y SIN ESCALAR) guardado con éxito en: {ruta_csv_fisica}!")
print(f"Dimensiones: {df_dataset_final.shape}")

# Calcular el Checksum (SHA-256) del archivo generado
sha256_hash = hashlib.sha256()
with open(ruta_csv_fisica, "rb") as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)

checksum_val = sha256_hash.hexdigest()
print(f"Checksum (SHA-256) calculado: {checksum_val}")

# Registrar o actualizar la huella en data/checksums.json.
ruta_checksums = raiz_proyecto / 'data' / 'checksums.json'
checksums_data = {}
if os.path.exists(ruta_checksums):
    try:
        with open(ruta_checksums, 'r', encoding='utf-8') as f:
            checksums_data = json.load(f)
    except Exception:
        checksums_data = {}

checksums_data[ruta_csv_key] = checksum_val

with open(ruta_checksums, 'w', encoding='utf-8') as f:
    json.dump(checksums_data, f, indent=4, ensure_ascii=False)

print(f"Checksum registrado correctamente en {ruta_checksums}")

In [ ]:
carpeta_procesados = raiz_proyecto / 'data' / 'processed'
carpeta_procesados.mkdir(parents=True, exist_ok=True)

# Armar el diccionario con la información del linaje y transformaciones
linaje_datos = {
    "proyecto": "Modelo Predictivo de Refrigeración - Planta Cerveza Tecate",
    "fecha_generacion": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "script_origen": "preprocesamiento.ipynb",
    "transformaciones_aplicadas": [
        "Agregación temporal de datos a nivel diario (cierre 23:59)",
        "Tratamiento de nulos e infinitos",
        "Target de Frío alineado con el cierre válido del día calendario D + 1",
        "Extracción de variables de calendario (Anio, Mes, Dia_Semana, Es_Fin_De_Semana)",
        "Creación de indicadores de eficiencia energética (Real vs Meta)",
        "Cálculo de ratios e interacciones específicas de frío y producción",
        "Integración de temperatura ambiente (media, max, min) de Tecate vía API Open-Meteo",
        "Correccion de outliers",
        "Correccion de Asimetrias mediante Yeo-Jhonson",
        "Estandarizacion de variables",
        f"Selección de características con SelectKBest (ANOVA F, f_regression) ajustado en Train: k = {len(features_finales)}",
        "Método elegido frente a RFE (Random Forest) y LASSO por MAE en TimeSeriesSplit, Validation y estabilidad (notebooks/seleccion_variables.ipynb)"
    ],
    "dimensiones_dataset_final": {
        "filas": int(df_diario.shape[0]),
        "columnas_totales": int(df_dataset_final.shape[1]),
        "features_seleccionadas": len(features_finales) if 'features_finales' in locals() else None
    }
}

ruta_archivo = carpeta_procesados / 'data_lineage.json'
with open(ruta_archivo, 'w', encoding='utf-8') as f:
    json.dump(linaje_datos, f, indent=4, ensure_ascii=False)

print(f" ¡Archivo de linaje creado y guardado con éxito en: {ruta_archivo}!")